# 02 Dense Event-Frame Feature Extraction Across the Full DAVIS Sequence

**Dataset:** Tonic `DAVISDATA` → `shapes_6dof.bag`  
**Sensor:** DAVIS 240 × 180  
**Windowing:** consecutive APS-frame timestamps (approximately 44.1 ms per interval)  
**Representation:** dense event-count frame  
**Feature extractor:** lightweight 3×3 local-difference / gradient-interest detector

### Objective
Run the dense event-frame representation over the **complete recording**, extract features independently in every APS-aligned event window, and quantify how often the representation provides useful spatial features.

> The notebook uses the **actual APS timestamps** from the dataset rather than hard-coding 44.1 ms. This keeps the experiment synchronized with the camera data.

## 1. Imports and configuration

No package installation is performed. The notebook assumes the same environment used for the existing Tonic/DAVIS notebook.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tonic

print("Tonic:", tonic.__version__)
print("NumPy:", np.__version__)
print("Current working directory:", Path.cwd())

## 2. Locate the existing DAVIS data

The expected project layout is:

```text
step_1_VIO/
├── data/DAVIS/DAVISDATA/shapes_6dof.bag
├── notebooks/
└── ...
```

The search below also works when the notebook is launched from the project root or another subdirectory.

In [ ]:
PROJECT_ROOT = None

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "data" / "DAVIS" / "DAVISDATA" / "shapes_6dof.bag").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    # Fallback to the known project location from the current setup.
    fallback = Path.home() / "Desktop/MVBC/Research_Work/06_Experiments/SNN/step_1_VIO"
    if (fallback / "data" / "DAVIS" / "DAVISDATA" / "shapes_6dof.bag").exists():
        PROJECT_ROOT = fallback

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/DAVIS/DAVISDATA/shapes_6dof.bag. "
        "Run this notebook from the project or edit PROJECT_ROOT manually."
    )

DATA_DIR = PROJECT_ROOT / "data" / "DAVIS"
BAG_PATH = DATA_DIR / "DAVISDATA" / "shapes_6dof.bag"
OUTPUT_DIR = PROJECT_ROOT / "results" / "dense_event_feature_extraction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bag:", BAG_PATH)
print("Output directory:", OUTPUT_DIR)

## 3. Load `shapes_6dof`

This follows the same Tonic dataset convention used in the existing notebook.

In [ ]:
dataset = tonic.datasets.DAVISDATA(
    save_to=str(DATA_DIR),
    recording="shapes_6dof"
)

print(dataset)
print("Number of dataset samples:", len(dataset))

data, target = dataset[0]
events = data[0]
imu = data[1]
aps = data[2]

print("Events:", events.shape, events.dtype)
print("IMU keys:", list(imu.keys()))
print("APS keys:", list(aps.keys()))
print("Target keys:", list(target.keys()))

## 4. Inspect event and APS timing

In [ ]:
t = events["t"].astype(np.int64)
x = events["x"].astype(np.int32)
y = events["y"].astype(np.int32)
p = events["p"]

frame_ts = np.asarray(aps["ts"]).astype(np.int64)
frames = aps.get("frames", None)

height, width = 180, 240

print(f"Event timestamp range: {t[0]} to {t[-1]} us")
print(f"Event duration: {(t[-1] - t[0]) / 1e6:.6f} s")
print(f"APS timestamp count: {len(frame_ts)}")
print(f"APS timestamp range: {frame_ts[0]} to {frame_ts[-1]} us")

if len(frame_ts) > 1:
    aps_dt_us = np.diff(frame_ts)
    print(f"Mean APS interval: {aps_dt_us.mean()/1000:.3f} ms")
    print(f"Median APS interval: {np.median(aps_dt_us)/1000:.3f} ms")
    print(f"Min/Max APS interval: {aps_dt_us.min()/1000:.3f} / {aps_dt_us.max()/1000:.3f} ms")

if frames is not None:
    print("APS frame array shape:", np.asarray(frames).shape)

## 5. Define APS-aligned event windows

Each interval is:

\[
W_k = [t^{APS}_k,\;t^{APS}_{k+1})
\]

This gives one dense event frame for each interval between consecutive APS frames. Windows with no events are retained so the statistics describe the **complete sequence**, not only active windows.

In [ ]:
# Keep only intervals that overlap the recorded event stream.
window_starts = frame_ts[:-1]
window_ends = frame_ts[1:]

valid = (window_ends > t[0]) & (window_starts < t[-1])
window_starts = window_starts[valid]
window_ends = window_ends[valid]

print("Total APS-aligned windows:", len(window_starts))
if len(window_starts):
    durations_ms = (window_ends - window_starts) / 1000.0
    print(f"Mean window duration: {durations_ms.mean():.3f} ms")
    print(f"Median window duration: {np.median(durations_ms):.3f} ms")
    print(f"Range: {durations_ms.min():.3f} to {durations_ms.max():.3f} ms")

## 6. Hardware-friendly feature extractor

For each dense event-count image, a 3×3 neighborhood is used. Four absolute directional differences are summed:

\[
S = |I_{x+1,y}-I_{x-1,y}| + |I_{x,y+1}-I_{x,y-1}|
+ |I_{x+1,y+1}-I_{x-1,y-1}| + |I_{x-1,y+1}-I_{x+1,y-1}|
\]

A feature is selected when `S > threshold`.

This is deliberately simple: additions, subtractions, absolute values, comparison, and local buffering are much easier to map to streaming FPGA/ASIC hardware than descriptor-heavy methods such as ORB.

In [ ]:
FEATURE_THRESHOLD = 4


def dense_event_frame(xw, yw, width=240, height=180):
    """Create a dense event-count frame for one time window."""
    frame = np.zeros((height, width), dtype=np.int32)
    if len(xw):
        np.add.at(frame, (yw, xw), 1)
    return frame


def extract_hw_features(frame, threshold=FEATURE_THRESHOLD):
    """Extract feature coordinates and integer interest scores from a 3x3 frame."""
    img = frame.astype(np.int32, copy=False)

    left   = img[1:-1, :-2]
    right  = img[1:-1, 2:]
    top    = img[:-2, 1:-1]
    bottom = img[2:, 1:-1]

    d1 = img[:-2, :-2]
    d2 = img[2:, 2:]
    d3 = img[2:, :-2]
    d4 = img[:-2, 2:]

    d_h  = np.abs(right - left)
    d_v  = np.abs(bottom - top)
    d_d1 = np.abs(d2 - d1)
    d_d2 = np.abs(d4 - d3)

    score = d_h + d_v + d_d1 + d_d2
    feature_mask = score > threshold

    ys, xs = np.where(feature_mask)
    scores = score[feature_mask].astype(np.int32)

    # Interior coordinates in the original 240x180 image.
    xs = xs + 1
    ys = ys + 1
    return xs.astype(np.int16), ys.astype(np.int16), scores


def spatial_coverage(xs, ys, width=240, height=180, grid_cols=12, grid_rows=9):
    """Percentage of coarse spatial cells containing at least one feature."""
    if len(xs) == 0:
        return 0.0
    gx = np.clip((xs * grid_cols) // width, 0, grid_cols - 1)
    gy = np.clip((ys * grid_rows) // height, 0, grid_rows - 1)
    occupied = np.unique(gy * grid_cols + gx)
    return 100.0 * len(occupied) / (grid_cols * grid_rows)

## 7. Process the complete sequence

Events are already sorted by timestamp, so `np.searchsorted` gives efficient window boundaries without scanning the full 17.9-million-event array for every window.

In [ ]:
window_results = []
feature_x = []
feature_y = []
feature_score = []

# Global event-index boundaries for each APS-aligned interval.
left_idx = np.searchsorted(t, window_starts, side="left")
right_idx = np.searchsorted(t, window_ends, side="left")

sequence_start = time.perf_counter()

for k, (ts, te, i0, i1) in enumerate(zip(window_starts, window_ends, left_idx, right_idx)):
    t0 = time.perf_counter()

    xw = x[i0:i1]
    yw = y[i0:i1]

    frame = dense_event_frame(xw, yw, width=width, height=height)
    xs, ys, scores = extract_hw_features(frame)

    active_pixels = int(np.count_nonzero(frame))
    event_count = int(i1 - i0)
    n_features = int(len(xs))

    result = {
        "window": k,
        "t_start_us": int(ts),
        "t_end_us": int(te),
        "duration_ms": float((te - ts) / 1000.0),
        "event_count": event_count,
        "active_pixels": active_pixels,
        "feature_count": n_features,
        "feature_density_per_kpixel": float(n_features / (width * height) * 1000.0),
        "spatial_coverage_percent": spatial_coverage(xs, ys, width, height),
        "mean_score": float(scores.mean()) if n_features else 0.0,
        "std_score": float(scores.std()) if n_features else 0.0,
        "max_score": int(scores.max()) if n_features else 0,
        "processing_ms": float((time.perf_counter() - t0) * 1000.0),
    }
    window_results.append(result)
    feature_x.append(xs)
    feature_y.append(ys)
    feature_score.append(scores)

    if (k + 1) % 100 == 0 or k == 0 or k == len(window_starts) - 1:
        print(
            f"Window {k+1:4d}/{len(window_starts)} | "
            f"events={event_count:6d} | features={n_features:4d} | "
            f"coverage={result['spatial_coverage_percent']:5.1f}%"
        )

sequence_elapsed = time.perf_counter() - sequence_start
summary_df = pd.DataFrame(window_results)

print("\nCompleted full-sequence extraction")
print(f"Windows processed: {len(summary_df)}")
print(f"Total processing time: {sequence_elapsed:.3f} s")
print(f"Mean processing time/window: {summary_df['processing_ms'].mean():.3f} ms")

## 8. Define and count meaningful event windows

A window should not be called useful merely because it contains a few detected points. For this first screening, a **meaningful window** is defined using configurable minimums:

- minimum number of extracted features
- minimum spatial coverage
- minimum event activity

These thresholds are experimental parameters, not final scientific claims. They can be changed later and the analysis rerun.

In [ ]:
MIN_FEATURES = 10
MIN_COVERAGE_PERCENT = 2.0
MIN_EVENTS = 20

summary_df["meaningful"] = (
    (summary_df["feature_count"] >= MIN_FEATURES) &
    (summary_df["spatial_coverage_percent"] >= MIN_COVERAGE_PERCENT) &
    (summary_df["event_count"] >= MIN_EVENTS)
)

meaningful_count = int(summary_df["meaningful"].sum())
meaningful_percent = 100.0 * meaningful_count / len(summary_df)

print(f"Meaningful windows: {meaningful_count} / {len(summary_df)}")
print(f"Meaningful-window percentage: {meaningful_percent:.2f}%")
print("\nFeature-count statistics:")
print(summary_df["feature_count"].describe())
print("\nEvent-count statistics:")
print(summary_df["event_count"].describe())

## 9. Full-sequence statistics

In [ ]:
print("=== FULL SEQUENCE ===")
print(f"Windows                 : {len(summary_df)}")
print(f"Event duration          : {(t[-1]-t[0])/1e6:.3f} s")
print(f"Total events            : {len(events):,}")
print(f"Mean events/window      : {summary_df.event_count.mean():.2f}")
print(f"Mean active pixels      : {summary_df.active_pixels.mean():.2f}")
print(f"Mean features/window    : {summary_df.feature_count.mean():.2f}")
print(f"Median features/window  : {summary_df.feature_count.median():.2f}")
print(f"Max features/window     : {summary_df.feature_count.max()}")
print(f"Mean spatial coverage   : {summary_df.spatial_coverage_percent.mean():.2f}%")
print(f"Meaningful windows      : {meaningful_count} ({meaningful_percent:.2f}%)")
print(f"Mean extractor time     : {summary_df.processing_ms.mean():.4f} ms/window")

## 10. Plot event activity and feature count over the complete sequence

In [ ]:
time_s = summary_df["t_start_us"] / 1e6

plt.figure(figsize=(12, 5))
plt.plot(time_s, summary_df["event_count"], linewidth=1)
plt.xlabel("Time (s)")
plt.ylabel("Events / APS interval")
plt.title("Event Activity Across the Complete DAVIS Sequence")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(time_s, summary_df["feature_count"], linewidth=1)
plt.axhline(MIN_FEATURES, linestyle="--", label=f"minimum = {MIN_FEATURES}")
plt.xlabel("Time (s)")
plt.ylabel("Extracted features")
plt.title("Hardware-Friendly Feature Count Across the Complete Sequence")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(time_s, summary_df["spatial_coverage_percent"], linewidth=1)
plt.axhline(MIN_COVERAGE_PERCENT, linestyle="--", label=f"minimum = {MIN_COVERAGE_PERCENT}%")
plt.xlabel("Time (s)")
plt.ylabel("Spatial coverage (%)")
plt.title("Feature Spatial Coverage Across the Complete Sequence")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Distribution of extracted features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(summary_df["feature_count"], bins=40)
axes[0].set_xlabel("Features per event frame")
axes[0].set_ylabel("Number of windows")
axes[0].set_title("Feature Count Distribution")

axes[1].hist(summary_df["spatial_coverage_percent"], bins=40)
axes[1].set_xlabel("Spatial coverage (%)")
axes[1].set_ylabel("Number of windows")
axes[1].set_title("Spatial Coverage Distribution")

plt.tight_layout()
plt.show()

## 12. Visualize representative windows

The notebook selects windows from different activity levels rather than only showing the first frame. This helps verify that the detector behaves sensibly throughout the sequence.

In [ ]:
def show_window(k, figsize=(12, 5)):
    i0, i1 = int(left_idx[k]), int(right_idx[k])
    frame = dense_event_frame(x[i0:i1], y[i0:i1], width=width, height=height)
    xs, ys, scores = feature_x[k], feature_y[k], feature_score[k]

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(frame, cmap="gray", origin="upper")
    if len(xs):
        ax.scatter(xs, ys, s=12, facecolors="none", edgecolors="red", linewidths=0.7)
    ax.set_title(
        f"Window {k} | {summary_df.loc[k, 't_start_us']/1e6:.3f}–{summary_df.loc[k, 't_end_us']/1e6:.3f} s | "
        f"events={len(x[i0:i1])} | features={len(xs)}"
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()
    plt.show()

# First, median-feature, and highest-feature windows.
example_indices = sorted(set([
    0,
    int(summary_df["feature_count"].idxmin()),
    int(summary_df["feature_count"].median()),
    int(summary_df["feature_count"].idxmax()),
    len(summary_df) - 1,
]))

for k in example_indices:
    show_window(k)

## 13. Visualize selected meaningful windows

In [ ]:
meaningful_indices = np.flatnonzero(summary_df["meaningful"].to_numpy())

if len(meaningful_indices):
    # Show up to five approximately uniformly distributed meaningful windows.
    selected = meaningful_indices[np.linspace(0, len(meaningful_indices)-1, min(5, len(meaningful_indices))).astype(int)]
    print("Selected meaningful windows:", selected.tolist())
    for k in selected:
        show_window(int(k))
else:
    print("No windows satisfy the current meaningful-window thresholds.")
    print("Try lowering MIN_FEATURES, MIN_COVERAGE_PERCENT, or MIN_EVENTS for exploration.")

## 14. Save numerical results

Two files are saved:

- `window_statistics.csv`: one row per APS-aligned event window
- `feature_points.npz`: variable-length feature coordinates and scores for every window

This keeps the experiment reproducible without storing every dense 240×180 image.

In [ ]:
csv_path = OUTPUT_DIR / "window_statistics.csv"
npz_path = OUTPUT_DIR / "feature_points.npz"

summary_df.to_csv(csv_path, index=False)

np.savez_compressed(
    npz_path,
    window_starts_us=window_starts,
    window_ends_us=window_ends,
    feature_x=np.array(feature_x, dtype=object),
    feature_y=np.array(feature_y, dtype=object),
    feature_score=np.array(feature_score, dtype=object),
)

print("Saved:", csv_path)
print("Saved:", npz_path)

## 15. Optional: save a compact experiment configuration

In [ ]:
config = {
    "dataset": "DAVISDATA/shapes_6dof",
    "sensor_resolution": [width, height],
    "window_definition": "consecutive APS timestamps [t_k, t_{k+1})",
    "feature_detector": "3x3 four-direction absolute-difference score",
    "feature_threshold": FEATURE_THRESHOLD,
    "meaningful_min_features": MIN_FEATURES,
    "meaningful_min_coverage_percent": MIN_COVERAGE_PERCENT,
    "meaningful_min_events": MIN_EVENTS,
    "total_windows": int(len(summary_df)),
    "meaningful_windows": meaningful_count,
    "meaningful_percentage": meaningful_percent,
}

config_df = pd.DataFrame([config])
config_path = OUTPUT_DIR / "experiment_config.csv"
config_df.to_csv(config_path, index=False)
print("Saved:", config_path)
config_df.T

# Interpretation and next step

At this point we have a complete event-frame feature-extraction baseline over the entire DAVIS recording.

### The important outputs for the research study

1. **Feature availability:** how many APS-aligned windows contain useful features?
2. **Feature density:** how many candidate points are available per 44 ms interval?
3. **Spatial coverage:** are points distributed across the image or concentrated in a small region?
4. **Temporal stability:** do the same spatial features survive from one event frame to the next?
5. **Hardware cost:** the detector uses local pixel accesses, integer differences, absolute values, additions, and a threshold.

### Next experiment

The natural next notebook is **feature tracking across consecutive event frames**. For each feature in window `k`, find a feature in window `k+1` within a configurable spatial radius. Then measure:

- matched features/window
- match ratio
- displacement `(dx, dy)`
- displacement magnitude
- track lifetime
- feature loss rate
- trajectories over time

After that, the same event features can be associated with **APS-frame features**, which is the more important experiment for the eventual VIO/SLAM architecture.